In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers datasets torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 126.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 105.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nv

In [1]:
import json
with open('/content/db_sql_s1.json') as f:
    raw_data = json.load(f)

In [2]:
from datasets import Dataset
raw_data = Dataset.from_list(raw_data)
split_dataset = raw_data.train_test_split(test_size=0.2, seed=42)

In [3]:
train_dataset = split_dataset['train']
val_dataset = split_dataset['test']

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

Train size: 314
Validation size: 79


In [4]:
# Remove 'instruction' from both datasets
train_dataset = train_dataset.remove_columns(['instruction'])
val_dataset = val_dataset.remove_columns(['instruction'])

In [5]:
print(train_dataset.column_names)

['input', 'output']


In [6]:
doctypes = sorted(list(set([record['output'] for record in raw_data])))

label2id = {label: idx for idx, label in enumerate(doctypes)}
id2label = {idx: label for label, idx in label2id.items()}
def encode_labels(example):
    example['label'] = label2id[example['output']]
    return example

train_dataset = train_dataset.map(encode_labels)
val_dataset = val_dataset.map(encode_labels)
print(doctypes)

Map:   0%|          | 0/314 [00:00<?, ? examples/s]

Map:   0%|          | 0/79 [00:00<?, ? examples/s]

['BOM', 'Contact', 'Customer', 'Customer, Sales Order', 'Delivery Note', 'Delivery Note, Delivery Note Item', 'Department', 'Department, Employee', 'Employee', 'GL Entry', 'Item', 'Item, Item Tax', 'Journal Entry', 'Journal Entry Account', 'Journal Entry, Journal Entry Account', 'Lead', 'Operation', 'Opportunity', 'Payment Entry', 'Payment Entry Reference, Payment Entry', 'Project', 'Purchase Invoice', 'Purchase Order', 'Purchase Order, Purchase Order Item', 'Purchase Receipt', 'Purchase Receipt, Purchase Receipt Item', 'Quotation', 'Routing', 'Sales Invoice', 'Sales Invoice, Customer', 'Sales Invoice, Sales Invoice Item', 'Sales Order', 'Sales Order, Sales Order Item', 'Sales Partner', 'Stock Entry', 'Stock Entry, Stock Entry Detail', 'Stock Ledger Entry', 'Supplier', 'Supplier, Purchase Invoice', 'Task', 'Tax Withheld Vouchers', 'Warehouse', 'Warehouse, Stock Ledger Entry']


In [7]:
import csv

# Save to CSV format
with open("doctype_mapping.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["Doctype", "ID"])  # Header
    for label, idx in label2id.items():
        writer.writerow([label, idx])

print("Saved as doctype_mapping.csv")

Saved as doctype_mapping.csv


In [8]:
# As for now we are not working on the multi doctypes questions, roberto can be trained on that case later to predict multi labels.
def is_single_doctype(example):
    output = example['output']
    # If output is a list (bad), or output contains ',' or ' and '
    if isinstance(output, list):
        return False
    if ',' in output or ' and ' in output.lower():
        return False
    return True

# Filter train and val datasets
train_dataset = train_dataset.filter(is_single_doctype)
val_dataset = val_dataset.filter(is_single_doctype)

# Check sizes after cleaning
print("Train size after cleaning:", len(train_dataset))
print("Validation size after cleaning:", len(val_dataset))

Filter:   0%|          | 0/314 [00:00<?, ? examples/s]

Filter:   0%|          | 0/79 [00:00<?, ? examples/s]

Train size after cleaning: 275
Validation size after cleaning: 69


In [9]:
train_dataset[4:9]

{'input': ['How many sales orders have been partially billed?',
  'How many employees have retired already?',
  'Which item has the highest standard rate among stock items?',
  'How many cash invoices were created today?',
  'How many invoices were cancelled this quarter?'],
 'output': ['Sales Order',
  'Employee',
  'Item',
  'Sales Invoice',
  'Sales Invoice'],
 'label': [31, 8, 10, 28, 28]}

In [10]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained('hyrinmansoor/text2frappe-s1-roberta')

def preprocess_function(examples):
    return tokenizer(examples['input'], truncation=True, padding="max_length")

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/275 [00:00<?, ? examples/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

In [11]:
from huggingface_hub import login

login()

In [15]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification

model_name = "hyrinmansoor/text2frappe-s1-roberta"  # can be swapped anytime
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [12]:
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Changai/S1/Model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=2,
    save_strategy="epoch",
    report_to="none"
)

# Load the model with the correct number of labels and mappings
model_name = "hyrinmansoor/text2frappe-s1-roberta"
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(doctypes), # Pass the number of labels
    id2label=id2label,       # Pass the id2label mapping
    label2id=label2id ,
    ignore_mismatched_sizes=True# Pass the label2id mapping
)

# Create Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val
)

# Train the model
trainer.train()

# Evaluate the model
trainer.evaluate()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at hyrinmansoor/text2frappe-s1-roberta and are newly initialized because the shapes did not match:
- classifier.out_proj.bias: found shape torch.Size([814]) in the checkpoint and torch.Size([43]) in the model instantiated
- classifier.out_proj.weight: found shape torch.Size([814, 768]) in the checkpoint and torch.Size([43, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,No log,2.146202
2,No log,1.607609
3,No log,1.356354
4,No log,1.235582
5,No log,1.202254


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.1

{'eval_loss': 1.202254295349121,
 'eval_runtime': 0.4919,
 'eval_samples_per_second': 140.279,
 'eval_steps_per_second': 10.165,
 'epoch': 5.0}

In [13]:
model.save_pretrained("hyrinmansoor/text2frappe-s1-roberta")
tokenizer.save_pretrained("hyrinmansoor/text2frappe-s1-roberta")

('hyrinmansoor/text2frappe-s1-roberta/tokenizer_config.json',
 'hyrinmansoor/text2frappe-s1-roberta/special_tokens_map.json',
 'hyrinmansoor/text2frappe-s1-roberta/vocab.json',
 'hyrinmansoor/text2frappe-s1-roberta/merges.txt',
 'hyrinmansoor/text2frappe-s1-roberta/added_tokens.json')

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from huggingface_hub import create_repo

create_repo("text2frappe-s1", private=True)

In [ ]:
from huggingface_hub import upload_folder

upload_folder(
    repo_id="hyrinmansoor/text2frappe-s1-roberta",
    folder_path="/content/drive/MyDrive/Changai/S1/Model",
    path_in_repo=".",
    repo_type="model"
)


In [ ]:
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
# Change the model_path to the directory where the model was actually saved
model_path = "/content/drive/MyDrive/Changai/S1/Model"

# Add local_files_only=True to explicitly load from the local path
model = RobertaForSequenceClassification.from_pretrained(model_path, local_files_only=True)
tokenizer = RobertaTokenizerFast.from_pretrained(model_path, local_files_only=True)

In [16]:
import torch
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
test_data = [
    {"question": "How many sales returns were recorded this quarter?", "real_answer": "Sales Invoice"},
    {"question": "What is the total revenue from sales invoices this year?", "real_answer": "Sales Invoice"},
    {"question": "Which customer has the highest total invoice amount this year?", "real_answer": "Sales Invoice"},
    {"question": "How many invoices were submitted last week?", "real_answer": "Sales Invoice"},
    {"question": "What is the average invoice amount this month?", "real_answer": "Sales Invoice"},
    {"question": "List all invoices created today.", "real_answer": "Sales Invoice"},
    {"question": "What is the status of sales invoice ACC-SINV-2025-00005?", "real_answer": "Sales Invoice"},
    {"question": "List all sales invoices with their status.", "real_answer": "Sales Invoice"},
    {"question": "What is the total outstanding amount for all sales invoices?", "real_answer": "Sales Invoice"},
    {"question": "How many sales invoices were created last month?", "real_answer": "Sales Invoice"}

]


id2label = {str(k): v for k, v in id2label.items()}

results = []

for record in test_data:
    test_question = record["question"]
    real_answer = record["real_answer"]

    inputs = tokenizer(test_question, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        predicted_class_id = logits.argmax(dim=-1).item()

    predicted_doctype = id2label[str(predicted_class_id)]

    # Store
    results.append({
        "Question": test_question,
        "Real Answer": real_answer,
        "Model Prediction": predicted_doctype,
        "Correct?": "✅" if predicted_doctype == real_answer else "❌"
    })
df_results = pd.DataFrame(results)
print(df_results)

                                            Question    Real Answer  \
0  How many sales returns were recorded this quar...  Sales Invoice   
1  What is the total revenue from sales invoices ...  Sales Invoice   
2  Which customer has the highest total invoice a...  Sales Invoice   
3        How many invoices were submitted last week?  Sales Invoice   
4     What is the average invoice amount this month?  Sales Invoice   
5                   List all invoices created today.  Sales Invoice   
6  What is the status of sales invoice ACC-SINV-2...  Sales Invoice   
7         List all sales invoices with their status.  Sales Invoice   
8  What is the total outstanding amount for all s...  Sales Invoice   
9   How many sales invoices were created last month?  Sales Invoice   

  Model Prediction Correct?  
0    Sales Invoice        ✅  
1    Sales Invoice        ✅  
2    Sales Invoice        ✅  
3    Sales Invoice        ✅  
4    Sales Invoice        ✅  
5    Sales Invoice        ✅  
6    Sal

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
